## Representation learning baselines

Frozen **wav2vec 2.0**, **HuBERT**, and **Whisper** (encoder only) embeddings with a linear probe (**StandardScaler + LogisticRegression**) and **GroupKFold** grouped like the other model notebooks (`file_stem` for Androids, `participant_id` for RADAR). Metrics are saved under `results/metrics/repr_learn/`.

**Dependencies**: install once in a terminal (`pip install -r requirements.txt` or `pip install "transformers>=4.40,<5"`) so you get **transformers 4.x** (v5 is untested here). **Do not run `pip install` inside this notebook:** upgrading packages in a live Jupyter kernel often **crashes the kernel on Windows** because NumPy/Torch DLLs get out of sync. After any env change, use **Restart Kernel** before running model code.

If the kernel still dies: set `FORCE_CPU = True` in the code cell (GPU VRAM), set `BATCH_SIZE = 1`, and close other GPU apps.

**RADAR audio**: the processed RADAR CSV only stores `File` (e.g. `20201230_1100-scripted-1-1.wav`). Set `RADAR_AUDIO_ROOT` below to a folder whose tree contains those `.wav` files (files are resolved by name). If wavs are not available, the RADAR block will skip missing files automatically.

**Cell 2 (below)** trains a **small PyTorch MLP head** on pooled pretrained **Wav2Vec2 or HuBERT** frame outputs (encoder **weights frozen**; only the head updates). Requires **cell 1** to have been run first (shared `DEVICE`, `load_waveform_mono`, `MODEL_IDS`, etc.). Results: `repr_learn/androids_*_dl_head_cv_folds.csv`.

In [1]:
from __future__ import annotations

import gc
import os
from pathlib import Path
from typing import Callable

# Avoid oversubscribing CPU threads (occasionally unstable in Jupyter on Windows)
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

import librosa
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (
    HubertModel,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
    WhisperModel,
    WhisperProcessor,
)

# ---------------------------------------------------------------------------
# Paths (match other notebooks)
# ---------------------------------------------------------------------------
PROJECT = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project")
ANDROID_CSV = PROJECT / "data/processed/androids_model_dataset_basic.csv"
RADAR_CSV = PROJECT / "data/processed/radar_model_dataset_raw_features.csv"
RESULTS_PATH = PROJECT / "results/metrics/repr_learn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Point this at the directory tree that holds RADAR WAVs (recursive lookup by File name).
# Example guesses — adjust until `Path.exists()` resolves your data:
RADAR_AUDIO_ROOT = PROJECT / "datasets/RADAR-MDD"

# --- stability (kernel dies on GPU OOM / Windows DLL mismatch after pip in-notebook) ---
FORCE_CPU = False  # True if CUDA crashes or you need deterministic CPU-only runs
BATCH_SIZE = 1   # increase to 2–4 only if GPU memory allows

torch.set_num_threads(min(8, max(1, os.cpu_count() or 1)))
if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
else:
    DEVICE = torch.device("cpu")

TARGET_SR = 16000
MAX_AUDIO_SECONDS = 30
N_FOLDS = 5

MODEL_IDS = {
    "wav2vec2": "facebook/wav2vec2-base",
    "hubert": "facebook/hubert-base-ls960",
    "whisper": "openai/whisper-base",
}


def _mean_pool(hidden: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:
    if mask is None:
        return hidden.mean(dim=1)
    m = mask.unsqueeze(-1).to(hidden.dtype)
    summed = (hidden * m).sum(dim=1)
    denom = m.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def build_wav_filename_index(audio_root: Path) -> dict[str, Path]:
    if not audio_root.exists():
        return {}
    idx: dict[str, Path] = {}
    for p in audio_root.rglob("*.wav"):
        idx.setdefault(p.name, p)
        idx.setdefault(p.name.lower(), p)
    return idx


def load_waveform_mono(path: Path, target_sr: int) -> np.ndarray:
    wav, sr = librosa.load(str(path), sr=target_sr, mono=True)
    max_len = int(MAX_AUDIO_SECONDS * target_sr)
    if len(wav) > max_len:
        wav = wav[:max_len]
    return wav.astype(np.float32)


def encode_wav2vec_family(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    bs = BATCH_SIZE if batch_size is None else batch_size
    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
    if "hubert" in model_id.lower():
        model = HubertModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    else:
        model = Wav2Vec2Model.from_pretrained(model_id, low_cpu_mem_usage=True)
    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []
    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]
        feats = processor(
            waves,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )
        feats = {k: v.to(DEVICE) for k, v in feats.items()}
        with torch.inference_mode():
            outputs = model(**feats)
        pooled = _mean_pool(outputs.last_hidden_state, feats.get("attention_mask"))
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return np.vstack(out_list)


def encode_whisper(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    bs = BATCH_SIZE if batch_size is None else batch_size
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []
    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]
        inputs = processor(
            waves,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )
        input_features = inputs.input_features.to(DEVICE)
        with torch.inference_mode():
            enc = model.encoder(input_features)
            hidden = enc.last_hidden_state
        pooled = hidden.mean(dim=1)
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return np.vstack(out_list)


ENCODERS: dict[str, Callable[[list[Path], str, int], np.ndarray]] = {
    "wav2vec2": encode_wav2vec_family,
    "hubert": encode_wav2vec_family,
    "whisper": encode_whisper,
}


def align_embeddings(paths: pd.Series, unique_paths: np.ndarray, mat: np.ndarray) -> np.ndarray:
    lookup = {str(p): i for i, p in enumerate(unique_paths)}
    idx = np.array([lookup[str(p)] for p in paths], dtype=np.int64)
    return mat[idx]


def run_group_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    n_folds: int = N_FOLDS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    gkf = GroupKFold(n_splits=n_folds)
    fold_rows: list[dict] = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",
                        solver="lbfgs",
                        random_state=42,
                    ),
                ),
            ]
        )
        pipe.fit(X[tr], y[tr])

        pred = pipe.predict(X[te])
        proba = pipe.predict_proba(X[te])[:, 1]

        dummy = DummyClassifier(strategy="stratified", random_state=42)
        dummy.fit(X[tr], y[tr])
        p_dummy = dummy.predict_proba(X[te])[:, 1]

        e_m = np.abs(y[te] - proba)
        e_d = np.abs(y[te] - p_dummy)
        if len(e_m) >= 2 and not np.allclose(e_m, e_d):
            w_p = float(wilcoxon(e_m, e_d, zero_method="wilcox", mode="auto").pvalue)
        else:
            w_p = float("nan")

        fold_rows.append(
            {
                "fold": fold,
                "n_train": len(tr),
                "n_test": len(te),
                "accuracy": accuracy_score(y[te], pred),
                "f1": f1_score(y[te], pred, zero_division=0),
                "roc_auc": roc_auc_score(y[te], proba),
                "wilcoxon_p_vs_stratified_dummy": w_p,
            }
        )

    folds_df = pd.DataFrame(fold_rows)
    summary = pd.DataFrame(
        [
            {
                "subset": "all",
                "n_rows": len(y),
                "n_groups": len(np.unique(groups)),
                "accuracy_mean": folds_df["accuracy"].mean(),
                "accuracy_std": folds_df["accuracy"].std(),
                "f1_mean": folds_df["f1"].mean(),
                "f1_std": folds_df["f1"].std(),
                "roc_auc_mean": folds_df["roc_auc"].mean(),
                "roc_auc_std": folds_df["roc_auc"].std(),
            }
        ]
    )
    return folds_df, summary


def run_repr_for_dataset(
    dataset_key: str,
    manifest: pd.DataFrame,
    path_series: pd.Series,
    groups: pd.Series,
    y: pd.Series,
) -> None:
    valid = (
        path_series.notna()
        & groups.notna()
        & y.notna()
        & path_series.astype(str).str.len().gt(0)
    )
    df = manifest.loc[valid].copy()
    paths = path_series.loc[valid].astype(str)
    grp = groups.loc[valid].astype(str).values
    yt = y.loc[valid].astype(int).values

    uniq = pd.unique(paths)
    uniq_paths = np.array([Path(p) for p in uniq])

    print(f"{dataset_key}: {len(paths)} usable rows | {len(uniq_paths)} unique audio files")

    for model_name, encoder in ENCODERS.items():
        print(f"  -> embeddings: {model_name} ({MODEL_IDS[model_name]})")
        emb_uniq = encoder(uniq_paths.tolist(), MODEL_IDS[model_name])
        X = align_embeddings(paths, uniq, emb_uniq)

        folds_df, summary_df = run_group_cv(X, yt, grp)

        folds_path = RESULTS_PATH / f"{dataset_key}_{model_name}_cv_folds.csv"
        summary_path = RESULTS_PATH / f"{dataset_key}_{model_name}_summary.csv"
        folds_df.to_csv(folds_path, index=False)
        summary_df.to_csv(summary_path, index=False)
        print(f"     saved: {folds_path.name}, {summary_path.name}")


def prepare_androids() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    df = pd.read_csv(ANDROID_CSV)
    df["file_path"] = df["file_path"].astype(str).str.strip()
    df["file_stem"] = df["file_stem"].astype(str).str.strip()
    df["depressed"] = pd.to_numeric(df["depressed"], errors="coerce")
    path_series = df["file_path"].map(lambda p: Path(p))
    exists = path_series.map(lambda p: p.is_file())
    df = df.loc[exists].copy()
    path_series = path_series.loc[exists]
    return df, path_series, df["file_stem"], df["depressed"]


def prepare_radar(wav_index: dict[str, Path]) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series] | None:
    df = pd.read_csv(RADAR_CSV)
    df["participant_id"] = df["participant_id"].astype(str).str.strip()
    df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")
    df = df.dropna(subset=["phq8_score", "File", "participant_id"]).copy()
    df["depressed"] = (df["phq8_score"] >= 10).astype(int)
    names = df["File"].astype(str).str.strip()

    def resolve(name: str) -> Path | None:
        if not name:
            return None
        if name in wav_index:
            return wav_index[name]
        low = name.lower()
        return wav_index.get(low)

    audio_paths = names.map(resolve)
    n_ok = audio_paths.notna().sum()
    if n_ok == 0:
        print(
            "RADAR: could not resolve any WAV paths. Set RADAR_AUDIO_ROOT so it contains",
            "the recording files referenced in column 'File'.",
        )
        return None

    df = df.loc[audio_paths.notna()].copy()
    paths_series = audio_paths.loc[audio_paths.notna()].map(lambda p: Path(p))
    if n_ok < len(names):
        print(f"RADAR: using {n_ok} / {len(names)} rows with found audio files")

    return df, paths_series, df["participant_id"], df["depressed"]


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
android_df, a_paths, a_groups, a_y = prepare_androids()
run_repr_for_dataset("androids", android_df, a_paths, a_groups, a_y)

RADAR_IDX = build_wav_filename_index(RADAR_AUDIO_ROOT)
rad = prepare_radar(RADAR_IDX)
if rad is not None:
    radar_df, r_paths, r_groups, r_y = rad
    run_repr_for_dataset("radar", radar_df, r_paths, r_groups, r_y)

print("Done. Outputs in:", RESULTS_PATH)

androids: 224 usable rows | 115 unique audio files
  -> embeddings: wav2vec2 (facebook/wav2vec2-base)


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

c:\Users\janku\Documents\KCL\Research Project\Research Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\janku\.cache\huggingface\hub\models--facebook--wav2vec2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

     saved: androids_whisper_cv_folds.csv, androids_whisper_summary.csv
RADAR: could not resolve any WAV paths. Set RADAR_AUDIO_ROOT so it contains the recording files referenced in column 'File'.
Done. Outputs in: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\repr_learn


In [1]:
# ---------------------------------------------------------------------------
# Simple DL: frozen Wav2Vec2 / HuBERT trunk + trainable 2-layer MLP head
# (Run cell 1 first so DEVICE, MODEL_IDS, load_waveform_mono, TARGET_SR, ... exist.)
# ---------------------------------------------------------------------------

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import wilcoxon
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset
from transformers import HubertModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model

try:
    DEVICE, MODEL_IDS, prepare_androids, load_waveform_mono, RESULTS_PATH, N_FOLDS, TARGET_SR
except NameError as e:
    raise RuntimeError(
        "Run the previous notebook cell first (imports + path setup)."
    ) from e

DL_BACKBONE = "wav2vec2"  # or "hubert"
DL_EPOCHS = 8
DL_BATCH = 4
DL_LR = 1e-3
DL_NUM_WORKERS = 0  # Windows + Jupyter: keep 0


def mean_pool_hidden(
    hidden: torch.Tensor, attention_mask: torch.Tensor | None
) -> torch.Tensor:
    if attention_mask is None:
        return hidden.mean(dim=1)
    m = attention_mask.unsqueeze(-1).to(dtype=hidden.dtype)
    summed = (hidden * m).sum(dim=1)
    denom = m.sum(dim=1).clamp(min=1e-6)
    return summed / denom


class FrozenEncoderClassifier(nn.Module):
    """Pooled CNN / transformer outputs -> small MLP -> logit."""

    def __init__(self, backbone_key: str) -> None:
        super().__init__()
        model_id = MODEL_IDS[backbone_key]
        if backbone_key == "hubert":
            self.encoder = HubertModel.from_pretrained(
                model_id, low_cpu_mem_usage=True
            )
        else:
            self.encoder = Wav2Vec2Model.from_pretrained(
                model_id, low_cpu_mem_usage=True
            )
        for p in self.encoder.parameters():
            p.requires_grad = False
        h = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(h, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.encoder(
            input_values,
            attention_mask=attention_mask,
        ).last_hidden_state
        pooled = mean_pool_hidden(out, attention_mask)
        return self.head(pooled).squeeze(-1)


class _AudioDS(Dataset):
    def __init__(self, path_strings: list[str], labels: np.ndarray) -> None:
        self.paths = path_strings
        self.y = labels.astype(np.float32)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int) -> tuple[np.ndarray, float]:
        wav = load_waveform_mono(Path(self.paths[i]), TARGET_SR)
        return wav, float(self.y[i])


def dl_collate(
    batch: list[tuple[np.ndarray, float]],
    processor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    waves, ys = zip(*batch)
    wave_list = list(waves)
    try:
        feats = processor(
            wave_list,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
    except TypeError:
        feats = processor(
            wave_list,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )
    input_values = feats["input_values"]
    if "attention_mask" in feats:
        attention_mask = feats["attention_mask"]
    else:
        # Some HF versions omit mask; build from raw lengths after padding to batch max T
        lengths = [len(w) for w in wave_list]
        b, t = input_values.shape
        attention_mask = torch.zeros(b, t, dtype=torch.long)
        for i, L in enumerate(lengths):
            L_eff = min(L, t)
            attention_mask[i, :L_eff] = 1
    y = torch.tensor(ys, dtype=torch.float32)
    return input_values, attention_mask, y


def train_one_epoch(
    model: FrozenEncoderClassifier,
    loader: DataLoader,
    opt: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> float:
    model.train()
    total, n = 0.0, 0
    for input_values, attn, yb in loader:
        input_values = input_values.to(DEVICE)
        attn = attn.to(DEVICE)
        yb = yb.to(DEVICE)
        opt.zero_grad()
        logits = model(input_values, attn)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        total += float(loss.detach().cpu()) * yb.size(0)
        n += yb.size(0)
    return total / max(n, 1)


@torch.inference_mode()
def eval_fold(
    model: FrozenEncoderClassifier,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    logits_all, y_all = [], []
    for input_values, attn, yb in loader:
        input_values = input_values.to(DEVICE)
        attn = attn.to(DEVICE)
        logits = model(input_values, attn).cpu().numpy()
        logits_all.append(logits)
        y_all.append(yb.numpy())
    return np.concatenate(logits_all), np.concatenate(y_all)


def run_dl_androids() -> None:
    df, paths, groups, y = prepare_androids()
    valid = paths.notna() & groups.notna() & y.notna()
    df = df.loc[valid].copy()
    path_arr = paths.loc[valid].astype(str).values
    y_arr = y.loc[valid].astype(int).values
    grp = groups.loc[valid].astype(str).values

    processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_IDS[DL_BACKBONE])
    gkf = GroupKFold(n_splits=N_FOLDS)
    fold_rows: list[dict] = []

    for fold, (tr_idx, te_idx) in enumerate(
        gkf.split(np.zeros(len(y_arr)), y_arr, groups=grp), start=1
    ):
        print(f"DL {DL_BACKBONE} fold {fold}/{N_FOLDS}")
        train_ds = _AudioDS(path_arr[tr_idx].tolist(), y_arr[tr_idx])
        test_ds = _AudioDS(path_arr[te_idx].tolist(), y_arr[te_idx])
        collate_fn = lambda b: dl_collate(b, processor)
        train_loader = DataLoader(
            train_ds,
            batch_size=DL_BATCH,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            collate_fn=collate_fn,
        )
        test_loader = DataLoader(
            test_ds,
            batch_size=DL_BATCH,
            shuffle=False,
            num_workers=DL_NUM_WORKERS,
            collate_fn=collate_fn,
        )

        model = FrozenEncoderClassifier(DL_BACKBONE).to(DEVICE)
        opt = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=DL_LR,
        )
        y_tr = y_arr[tr_idx]
        n_pos = (y_tr == 1).sum()
        n_neg = (y_tr == 0).sum()
        pos_weight = torch.tensor(
            [n_neg / max(int(n_pos), 1)], dtype=torch.float32, device=DEVICE
        )
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        for ep in range(1, DL_EPOCHS + 1):
            loss_tr = train_one_epoch(model, train_loader, opt, loss_fn)
            if ep == 1 or ep == DL_EPOCHS:
                print(f"   epoch {ep}/{DL_EPOCHS}  train_loss={loss_tr:.4f}")

        logits_te, y_te = eval_fold(model, test_loader)
        proba = 1.0 / (1.0 + np.exp(-logits_te))
        pred = (proba >= 0.5).astype(int)

        dummy = DummyClassifier(strategy="stratified", random_state=42)
        dummy.fit(np.zeros((len(tr_idx), 1)), y_tr)
        p_dummy = dummy.predict_proba(np.zeros((len(te_idx), 1)))[:, 1]

        e_m = np.abs(y_te - proba)
        e_d = np.abs(y_te - p_dummy)
        if len(e_m) >= 2 and not np.allclose(e_m, e_d):
            w_p = float(wilcoxon(e_m, e_d, zero_method="wilcox", mode="auto").pvalue)
        else:
            w_p = float("nan")

        fold_rows.append(
            {
                "fold": fold,
                "n_train": len(tr_idx),
                "n_test": len(te_idx),
                "accuracy": accuracy_score(y_te, pred),
                "f1": f1_score(y_te, pred, zero_division=0),
                "roc_auc": roc_auc_score(y_te, proba),
                "wilcoxon_p_vs_stratified_dummy": w_p,
            }
        )

        model.cpu()
        del model, opt, loss_fn, train_loader, test_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    folds_df = pd.DataFrame(fold_rows)
    summary_df = pd.DataFrame(
        [
            {
                "backbone": DL_BACKBONE,
                "n_rows": len(y_arr),
                "n_groups": len(np.unique(grp)),
                "epochs": DL_EPOCHS,
                "accuracy_mean": folds_df["accuracy"].mean(),
                "accuracy_std": folds_df["accuracy"].std(),
                "f1_mean": folds_df["f1"].mean(),
                "f1_std": folds_df["f1"].std(),
                "roc_auc_mean": folds_df["roc_auc"].mean(),
                "roc_auc_std": folds_df["roc_auc"].std(),
            }
        ]
    )
    tag = f"androids_{DL_BACKBONE}_dl_head"
    folds_df.to_csv(RESULTS_PATH / f"{tag}_cv_folds.csv", index=False)
    summary_df.to_csv(RESULTS_PATH / f"{tag}_summary.csv", index=False)
    print("Saved:", RESULTS_PATH / f"{tag}_cv_folds.csv")


run_dl_androids()

RuntimeError: Run the previous notebook cell first (imports + path setup).